Minimal setup - zeroshot deepforest

Data source: Settore innovazione digitale, dati, tecnologia e polo archivistico (Regione Emilia-Romagna), Bologna RGB 2023
License: CC BY 4.0 — https://creativecommons.org/licenses/by/4.0/
Changes: Cropped  and used for tree detection.

In [ ]:
# cell 1 — install
!pip install deepforest

# cell 2 — zero-shot detection on your ortho
from deepforest import main
import pandas as pd

m = main.deepforest()
m.load_model(model_name="weecology/deepforest-tree", revision="main")
m.config["score_thresh"] = 0.1  

predictions = m.predict_tile(
    path="/kaggle/input/datasets/tercasaskova/bologna-tiff/bologna_rgb_2023.tif",
    patch_size=400,
    patch_overlap=0.10,
    iou_threshold=0.15,
    dataloader_strategy="window",  # memory safe — never loads full image
)

predictions.to_csv("/kaggle/working/raw_predictions_checkpoint.csv", index=False)
print(predictions.head())
print(f"{len(predictions):,} detections")

finetuning deepforest on TCP - european sources(+ others) - well anotated

In [ ]:
# ── Cell 1: Install ─────────────────────────────────────────────────────────
!pip install deepforest datasets torch torchvision --quiet

# ── Cell 2: Load TCD, filter for European biomes ────────────────────────────
from datasets import load_dataset
import json, os, shutil
from PIL import Image
import pandas as pd
from pathlib import Path

ds = load_dataset("restor/tcd", split="train")

# European-ish biome ids (from the viewer: biome 4 = East European forest steppe, etc.)
# Filter broadly by lat/lon for Europe: lon 10–35, lat 45–60
european = ds.filter(
    lambda x: 10 <= (x["lon"] or 0) <= 40 and 40 <= (x["lat"] or 0) <= 65
)
print(f"European subset: {len(european)} tiles")

Filter:   0%|          | 0/4169 [00:00<?, ? examples/s]

In [ ]:
# ── Cell 3: Convert TCD COCO annotations → DeepForest CSV format ─────────────
# DeepForest expects: image_path, xmin, ymin, xmax, ymax, label
TILES_DIR = Path("/kaggle/working/tcd_tiles")
TILES_DIR.mkdir(exist_ok=True)

rows = []
for i, sample in enumerate(european):
    img_path = TILES_DIR / f"tile_{i:04d}.png"
    sample["image"].save(img_path)
    
    try:
        annots = json.loads(sample["coco_annotations"])
    except Exception:
        continue
    
    for ann in annots:
        cat = ann.get("category_id")
        if cat != 2:  # category 2 = individual tree (1 = canopy cover)
            continue
        if ann.get("iscrowd", 0):
            continue
        x, y, w, h = ann["bbox"]
        if w < 5 or h < 5:
            continue
        rows.append({
            "image_path": str(img_path),
            "xmin": x, "ymin": y,
            "xmax": x + w, "ymax": y + h,
            "label": "Tree"
        })

df = pd.DataFrame(rows)
df.to_csv("/kaggle/working/tcd_european_train.csv", index=False)
print(f"Training annotations: {len(df):,} boxes from {df['image_path'].nunique()} tiles")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.9/41.9 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.6/20.6 MB 78.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.1/588.1 kB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.6/471.6 kB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.4/217.4 kB 15.4 MB/s eta 0:00:00
[2026-04-23T08:43:56+0000  INFO] NumExpr defaulting to 4 threads.
[2026-04-23T08:43:58+0000  DEBUG] PyTorch version 2.9.0+cu126 available.
[2026-04-23T08:43:58+0000  DEBUG] Polars version 1.31.0 available.
[2026-04-23T08:43:58+0000  DEBUG] Duckdb version 1.3.2 available.
[2026-04-23T08:43:58+0000  INFO] TensorFlow version 2.19.0 available.
[2026-04-23T08:43:58+0000  INFO] JAX version 0.7.2 available.
[2026-04-23T08:43:59+0000  DEBUG] connect_tcp.started host='huggingface.co' port=443 local_address=Non

README.md: 0.00B [00:00, ?B/s]

[2026-04-23T08:43:59+0000  DEBUG] receive_response_body.started request=<Request [b'GET']>
[2026-04-23T08:43:59+0000  DEBUG] receive_response_body.complete
[2026-04-23T08:43:59+0000  DEBUG] response_closed.started
[2026-04-23T08:43:59+0000  DEBUG] response_closed.complete
[2026-04-23T08:43:59+0000  DEBUG] Attempting to release lock 136480217938736 on /root/.cache/huggingface/hub/.locks/datasets--restor--tcd/d801bbb2b28677b730e5330f0d105c5715074473.lock
[2026-04-23T08:43:59+0000  DEBUG] Lock 136480217938736 released on /root/.cache/huggingface/hub/.locks/datasets--restor--tcd/d801bbb2b28677b730e5330f0d105c5715074473.lock
[2026-04-23T08:43:59+0000  DEBUG] send_request_headers.started request=<Request [b'HEAD']>
[2026-04-23T08:43:59+0000  DEBUG] send_request_headers.complete
[2026-04-23T08:43:59+0000  DEBUG] send_request_body.started request=<Request [b'HEAD']>
[2026-04-23T08:43:59+0000  DEBUG] send_request_body.complete
[2026-04-23T08:43:59+0000  DEBUG] receive_response_headers.started r

[2026-04-23T08:43:59+0000  WARNING] Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
[2026-04-23T08:43:59+0000  DEBUG] connect_tcp.started host='datasets-server.huggingface.co' port=443 local_address=None timeout=100.0 socket_options=None
[2026-04-23T08:44:00+0000  DEBUG] connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7c20c6e981a0>
[2026-04-23T08:44:00+0000  DEBUG] start_tls.started ssl_context=<ssl.SSLContext object at 0x7c20c6aaca50> server_hostname='datasets-server.huggingface.co' timeout=100.0
[2026-04-23T08:44:00+0000  DEBUG] start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7c20c6e98c20>
[2026-04-23T08:44:00+0000  DEBUG] send_request_headers.started request=<Request [b'GET']>
[2026-04-23T08:44:00+0000  DEBUG] send_request_headers.complete
[2026-04-23T08:44:00+0000  DEBUG] send_request_body.started request=<Request [b'GET']>
[202

data/train-00000-of-00007.parquet:   0%|          | 0.00/464M [00:00<?, ?B/s]

...

data/train-00004-of-00007.parquet:   0%|          | 0.00/468M [00:00<?, ?B/s]

[2026-04-23T08:44:21+0000  DEBUG] Attempting to release lock 136479825393760 on /root/.cache/huggingface/hub/.locks/datasets--restor--tcd/d656d76c4943bc30a6abf42acba3733e46c23f87380c4ea9edd9f23c84b46b09.lock
[2026-04-23T08:44:21+0000  DEBUG] Lock 136479825393760 released on /root/.cache/huggingface/hub/.locks/datasets--restor--tcd/d656d76c4943bc30a6abf42acba3733e46c23f87380c4ea9edd9f23c84b46b09.lock
[2026-04-23T08:44:21+0000  DEBUG] send_request_headers.started request=<Request [b'HEAD']>
[2026-04-23T08:44:21+0000  DEBUG] send_request_headers.complete
[2026-04-23T08:44:21+0000  DEBUG] send_request_body.started request=<Request [b'HEAD']>
[2026-04-23T08:44:21+0000  DEBUG] send_request_body.complete
[2026-04-23T08:44:21+0000  DEBUG] receive_response_headers.started request=<Request [b'HEAD']>
[2026-04-23T08:44:21+0000  DEBUG] receive_response_headers.complete return_value=(b'HTTP/1.1', 302, b'Found', [(b'Content-Type', b'text/plain; charset=utf-8'), (b'Content-Length', b'1371'), (b'Conne

data/train-00005-of-00007.parquet:   0%|          | 0.00/444M [00:00<?, ?B/s]

[2026-04-23T08:44:25+0000  DEBUG] Attempting to release lock 136479825342976 on /root/.cache/huggingface/hub/.locks/datasets--restor--tcd/a441a767fd91286aa2db3a425b36131cd26e0e3a629350756f71b8a2dd7efbb9.lock
[2026-04-23T08:44:25+0000  DEBUG] Lock 136479825342976 released on /root/.cache/huggingface/hub/.locks/datasets--restor--tcd/a441a767fd91286aa2db3a425b36131cd26e0e3a629350756f71b8a2dd7efbb9.lock
[2026-04-23T08:44:25+0000  DEBUG] send_request_headers.started request=<Request [b'HEAD']>
[2026-04-23T08:44:25+0000  DEBUG] send_request_headers.complete
[2026-04-23T08:44:25+0000  DEBUG] send_request_body.started request=<Request [b'HEAD']>
[2026-04-23T08:44:25+0000  DEBUG] send_request_body.complete
[2026-04-23T08:44:25+0000  DEBUG] receive_response_headers.started request=<Request [b'HEAD']>
[2026-04-23T08:44:25+0000  DEBUG] receive_response_headers.complete return_value=(b'HTTP/1.1', 302, b'Found', [(b'Content-Type', b'text/plain; charset=utf-8'), (b'Content-Length', b'1373'), (b'Conne

data/train-00006-of-00007.parquet:   0%|          | 0.00/455M [00:00<?, ?B/s]

In [ ]:

# ── Cell 4: Fine-tune DeepForest ─────────────────────────────────────────────
from deepforest import main
import torch

model = main.deepforest()
model.load_model(model_name="weecology/deepforest-tree", revision="main")  # warm start

# Config tuning for dense European urban/forest trees
model.config["train"]["epochs"] = 15
model.config["train"]["lr"] = 1e-4
model.config["train"]["batch_size"] = 4
model.config["score_thresh"] = 0.3
model.config["nms_thresh"] = 0.05      # aggressive NMS for dense canopy

# Use GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Training on: {device}")

model.create_trainer(fast_dev_run=False)
model.trainer.fit(model)  # DeepForest handles the CSV internally via config

model.save_model("/kaggle/working/deepforest_tcd_eu.pt")
print("Model saved.")


In [6]:
# ── Cell 5: Run on your orthophoto ───────────────────────────────────────────
predictions = model.predict_tile(
    path="/kaggle/input/datasets/tercasaskova/bologna-tiff/bologna_rgb_2023.tif",
    patch_size=600,       # larger patch → better context for dense canopies
    patch_overlap=0.25,   # more overlap → better blob splitting at edges
    iou_threshold=0.1,
    dataloader_strategy="window",
)
predictions.to_csv("/kaggle/working/predictions_finetuned.csv", index=False)
print(f"{len(predictions):,} detections")

# ── Cell 6: BBox → polygons + watershed crown splitting ─────────────────────
import geopandas as gpd
from shapely.geometry import box
import rasterio
from rasterio.transform import rowcol
import numpy as np
from scipy import ndimage
from skimage.feature import peak_local_max
from skimage.segmentation import watershed

# Load ortho for CRS/transform
with rasterio.open("/kaggle/input/datasets/tercasaskova/bologna-tiff/bologna_rgb_2023.tif") as src:
    transform = src.transform
    crs = src.crs

def pixel_bbox_to_geo(row, transform):
    """Convert pixel bbox to geographic polygon."""
    x_min, y_max = transform * (row["xmin"], row["ymin"])
    x_max, y_min = transform * (row["xmax"], row["ymax"])
    return box(x_min, y_min, x_max, y_max)

gdf = gpd.GeoDataFrame(
    predictions,
    geometry=[pixel_bbox_to_geo(r, transform) for _, r in predictions.iterrows()],
    crs=crs
)

# Reproject to UTM for metric operations
gdf_utm = gdf.to_crs(epsg=32632)  # UTM zone 32N covers Bologna

# Filter by crown area (typical tree: 4–200 m²)
gdf_utm["area_m2"] = gdf_utm.geometry.area
gdf_utm = gdf_utm[(gdf_utm["area_m2"] > 4) & (gdf_utm["area_m2"] < 200)]

# Dissolve blobs and re-split with watershed (for merged detections)
def watershed_split_blob(geom, chm_band=None):
    """
    For a polygon that looks too large (likely a merged blob),
    use distance transform to split into individual crowns.
    Returns list of sub-polygons.
    """
    from rasterio.features import rasterize
    from shapely.ops import unary_union
    
    if geom.area < 25:  # small enough = single tree, skip
        return [geom]
    
    # Rasterise the polygon at 0.5m resolution for watershed
    bounds = geom.bounds
    res = 0.5
    w = int((bounds[2] - bounds[0]) / res) + 2
    h = int((bounds[3] - bounds[1]) / res) + 2
    
    from rasterio.transform import from_bounds
    local_transform = from_bounds(*bounds, w, h)
    mask = rasterize([geom], out_shape=(h, w), transform=local_transform).astype(bool)
    
    dist = ndimage.distance_transform_edt(mask)
    coords = peak_local_max(dist, min_distance=int(3/res), labels=mask)
    if len(coords) <= 1:
        return [geom]
    
    markers, _ = ndimage.label(np.zeros_like(mask, dtype=bool))
    for idx, (r, c) in enumerate(coords):
        markers[r, c] = idx + 1
    
    labels = watershed(-dist, markers, mask=mask)
    
    from rasterio.features import shapes
    from shapely.geometry import shape
    sub_polys = []
    for region_id in range(1, labels.max() + 1):
        region_mask = (labels == region_id).astype("uint8")
        for geom_shape, val in shapes(region_mask, transform=local_transform):
            if val == 1:
                sub_polys.append(shape(geom_shape))
    return sub_polys if sub_polys else [geom]

# Apply watershed splitting to large detections (likely blobs)
BLOB_THRESHOLD_M2 = 50  # crowns > 50 m² probably contain multiple trees

split_geoms = []
for _, row in gdf_utm.iterrows():
    if row["area_m2"] > BLOB_THRESHOLD_M2:
        split_geoms.extend(watershed_split_blob(row.geometry))
    else:
        split_geoms.append(row.geometry)

gdf_final = gpd.GeoDataFrame(geometry=split_geoms, crs=gdf_utm.crs)
gdf_final["area_m2"] = gdf_final.geometry.area
gdf_final = gdf_final[(gdf_final["area_m2"] > 3) & (gdf_final["area_m2"] < 180)]

# Back to WGS84 or keep UTM for QGIS
gdf_final_wgs = gdf_final.to_crs(epsg=4326)
gdf_final_wgs.to_file("/kaggle/working/single_trees_bologna.geojson", driver="GeoJSON")
print(f"Final: {len(gdf_final_wgs):,} individual tree polygons → single_trees_bologna.geojson")

['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', 'config', 'configurer']
